# Notebook 9 . Audit Kualitas Data

Buat ringkasan kolom kosong, nilai unik, duplikat, dan nilai yang tidak sesuai aturan sederhana. Notebook menghasilkan laporan ringkas serta baris yang perlu diperiksa; notebook tidak mengubah file sumber.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Kolom_Wajib = ["id"]        # Kolom yang wajib terisi
Kolom_Unik = ["id"]       # Kombinasi kolom yang seharusnya unik; [] jika tidak ada
Nilai_Diizinkan = {                # Kosongkan {} bila tidak perlu
    # "status": ["aktif", "nonaktif"],
}
Batas_Kosong_Persen = 20.0
Folder_Output = Path("Output")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Buat laporan dan daftar baris bermasalah

In [ ]:
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=Separator_CSV, dtype="string", encoding="utf-8-sig", keep_default_na=True)
df.columns = df.columns.astype(str).str.strip()
summary = pd.DataFrame({
    "kolom": df.columns,
    "nilai_kosong": [int(df[c].isna().sum()) for c in df.columns],
    "persen_kosong": [round(float(df[c].isna().mean()*100), 2) for c in df.columns],
    "nilai_unik": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values("persen_kosong", ascending=False)
issues = pd.Series(False, index=df.index)
for col in Kolom_Wajib:
    if col not in df: raise KeyError(f"Kolom wajib tidak ditemukan: {col}")
    issues |= df[col].isna() | df[col].astype("string").str.strip().eq("").fillna(False)
high_missing = [c for c in df.columns if df[c].isna().mean() * 100 > Batas_Kosong_Persen]
print(f"Kolom dengan nilai kosong di atas {Batas_Kosong_Persen}%: {high_missing if high_missing else 'tidak ada'}")
for col in high_missing: issues |= df[col].isna()
if Kolom_Unik:
    missing = [c for c in Kolom_Unik if c not in df]
    if missing: raise KeyError(f"Kolom kunci tidak ditemukan: {missing}")
    issues |= df.duplicated(Kolom_Unik, keep=False)
for col, allowed in Nilai_Diizinkan.items():
    if col not in df: raise KeyError(f"Kolom aturan tidak ditemukan: {col}")
    invalid = df[col].notna() & ~df[col].isin(allowed)
    print(f"{col}: {int(invalid.sum()):,} nilai di luar daftar yang diizinkan")
    issues |= invalid
print(f"Baris: {len(df):,}; kolom: {len(df.columns)}; baris perlu diperiksa: {int(issues.sum()):,}")
display(summary)


In [ ]:
outdir = Folder_Output if Folder_Output.is_absolute() else BASE_DIR / Folder_Output
outdir.mkdir(parents=True, exist_ok=True)
summary.to_csv(outdir / "ringkasan_kolom.csv", index=False, encoding="utf-8-sig", sep=Separator_CSV)
df.loc[issues].to_csv(outdir / "baris_perlu_diperiksa.csv", index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Laporan tersimpan di {outdir.resolve()}")